# 03. Data Preprocessing and Splits


In [1]:
from pathlib import Path
import numpy as np
import torch
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    load_cached_features,
    get_grouped_splits,
    create_dataloader,
    compute_class_weights,
    verify_zero_leakage,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Load Features

In [2]:
cache_path = ROOT / "data" / "processed" / "squat_pose_16f_v1.npz"
data = load_cached_features(cache_path)
X, y, groups = data["X"], data["y"], data["groups"]

print(f"X shape: {X.shape} (clips, frames, features)")
print(f"y shape: {y.shape} (0=bad: {(y==0).sum()}, 1=good: {(y==1).sum()})")
print(f"Total unique subjects: {len(np.unique(groups))}")

X shape: (150, 16, 51) (clips, frames, features)
y shape: (150,) (0=bad: 75, 1=good: 75)
Total unique subjects: 25


## 2. Participant Splits (Zero-Leakage)

In [3]:
train_idx, val_idx, test_idx = get_grouped_splits(X, y, groups, test_size=0.2, val_size=0.25, seed=SEED)

# Verify zero leakage
verify_zero_leakage(groups[train_idx], groups[val_idx], groups[test_idx])
print("Zero subject leakage verified across train, val, and test splits!\n")

for name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    labels = dict(zip(*np.unique(y[idx], return_counts=True)))
    subjs = sorted(set(groups[idx]))
    print(f"{name:12s}: {len(idx):3d} clips | Labels: {labels} | {len(subjs)} Subjects: {subjs}")

Zero subject leakage verified across train, val, and test splits!

Train       :  90 clips | Labels: {np.int64(0): np.int64(45), np.int64(1): np.int64(45)} | 15 Subjects: [np.str_('subject_002'), np.str_('subject_003'), np.str_('subject_004'), np.str_('subject_005'), np.str_('subject_006'), np.str_('subject_010'), np.str_('subject_011'), np.str_('subject_014'), np.str_('subject_015'), np.str_('subject_016'), np.str_('subject_018'), np.str_('subject_020'), np.str_('subject_021'), np.str_('subject_022'), np.str_('subject_023')]
Validation  :  30 clips | Labels: {np.int64(0): np.int64(15), np.int64(1): np.int64(15)} | 5 Subjects: [np.str_('subject_007'), np.str_('subject_008'), np.str_('subject_013'), np.str_('subject_019'), np.str_('subject_026')]
Test        :  30 clips | Labels: {np.int64(0): np.int64(15), np.int64(1): np.int64(15)} | 5 Subjects: [np.str_('subject_001'), np.str_('subject_009'), np.str_('subject_012'), np.str_('subject_017'), np.str_('subject_024')]


## 3. DataLoaders and Class Weights

In [4]:
train_loader = create_dataloader(X, y, train_idx, batch_size=32, shuffle=True)
val_loader   = create_dataloader(X, y, val_idx, batch_size=32, shuffle=False)
test_loader  = create_dataloader(X, y, test_idx, batch_size=32, shuffle=False)

weights = compute_class_weights(y[train_idx])
print(f"Class balancing weights: {weights.tolist()}")

sample_x, sample_y = next(iter(train_loader))
print(f"Batch tensor shape: {sample_x.shape} (batch_size, time_steps, features)")
print(f"Batch labels shape: {sample_y.shape}")

Class balancing weights: [1.0, 1.0]
Batch tensor shape: torch.Size([32, 16, 51]) (batch_size, time_steps, features)
Batch labels shape: torch.Size([32])
